# ANPR cascade: Colab GPU test

Runs the full pipeline on a Colab GPU. Order: setup, unit tests, OCR check with the fine-tuned weights, **vehicle-only live-stream benchmark** (works without plate weights), then the full pipeline once the plate detector is trained.

Before starting, put `anpr_cascade.tar.gz` in `MyDrive/bosch_anpr/code_backup/` and a test video on Drive.

## 1. Setup

In [ ]:
import os, subprocess, sys
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount('/content/drive')
B = '/content/drive/MyDrive/bosch_anpr'
VIDEO = B + '/data/videos/sample.mp4'      # <-- set to your test video
GT_CSV = B + '/data/videos/sample_plates.txt'  # one legible plate per line (optional)


In [ ]:
!pip -q install ultralytics rfdetr pyyaml opencv-python-headless
!rm -rf /content/anpr_cascade && tar xzf {B}/code_backup/anpr_cascade.tar.gz -C /content
!test -d /content/OpenOCR || git clone -q --depth 1 https://github.com/Topdu/OpenOCR.git /content/OpenOCR
sys.path.insert(0, '/content/anpr_cascade/src'); os.chdir('/content/anpr_cascade')


## 2. Unit tests (CPU, seconds)

In [ ]:
!pip -q install pytest ruff && python -m pytest -q && ruff check .

## 3. OCR check with the fine-tuned RepSVTR

Loads `best_ema.pt` and reads a synthetic noise crop to confirm the model builds, loads strictly (37-class head) and decodes. Point `CROP` at a real plate image to see a real read.

In [ ]:
import numpy as np
from anpr.ocr.recognizer import RepSvtrRecognizer, PlateReader
ocr = RepSvtrRecognizer('/content/OpenOCR', B + '/models/ocr_repsvtr_ft_v2/best_ema.pt', B + '/models/ocr_repsvtr_ft_v2/plate_dict.txt')
CROP = None  # e.g. B + '/some_plate.jpg'
import cv2
img = cv2.imread(CROP) if CROP else np.random.randint(0, 255, (48, 192, 3), dtype=np.uint8)
print(PlateReader(ocr)(img))


## 4. Vehicle-only live-stream benchmark

`realtime: true` paces the file at its source FPS, so it behaves like a camera. Read the output as: `frames_dropped` near 0 means the pipeline keeps up with real time on this GPU; `p95_ms` per stage shows where the time goes.

In [ ]:
!python -m anpr.run --source {VIDEO} --no-plates --jsonl /content/out/v.jsonl --max-frames 600

## 5. Full pipeline (needs trained plate weights)

Train the plate detector at the end, save `checkpoint_best_ema.pth` to `models/det_rfdetr_nano_main/` on Drive, then run this.

In [ ]:
!python -m anpr.run --config configs/default.yaml --source {VIDEO} --jsonl /content/out/full.jsonl --sqlite /content/out/full.db

## 6. Score against ground-truth plates

In [ ]:
from anpr.eval.pipeline_eval import load_events, evaluate_events
gt = [l.strip().upper() for l in open(GT_CSV) if l.strip()]
print(evaluate_events(load_events('/content/out/full.jsonl'), gt))